In [ ]:
import os 
os.chdir("/hpc/home/ephdh/workspace/suzhou_false_validation/analysis")

import ast 
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from typing import Tuple

In [ ]:
result_df = pd.read_csv(
        "/hpc/home/ephdh/workspace/suzhou_false_validation/data/meta_data/meta_data_with_model_scores.csv", 
        encoding="utf-16",  
        converters={"DICOMPaths": ast.literal_eval, 
                    'clip_image_scores': ast.literal_eval,
                    'cnn_image_scores': ast.literal_eval}
        )
result_df['DICOM_paths'] = result_df['DICOM_paths'].apply(ast.literal_eval)

In [ ]:
with pd.option_context('display.max_columns', None):
    display(result_df.head())

In [ ]:
result_df.columns.to_list()

In [ ]:
# ---------------------------------------------------------------------
# 1. BOOTSTRAP HELPERS
# ---------------------------------------------------------------------

def bootstrap_binary_metrics(y_true, y_pred, y_score=None, n_boot=2000, seed=1):
    """Bootstrapped CIs for sensitivity, specificity, and AUC (if score given)."""
    rng = np.random.default_rng(seed)
    n = len(y_true)

    sens_list, spec_list, auc_list = [], [], []

    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt = y_true[idx]
        yp = y_pred[idx]
        ys = y_score[idx] if y_score is not None else None

        tn, fp, fn, tp = confusion_matrix(yt, yp).ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan

        sens_list.append(sens)
        spec_list.append(spec)

        if ys is not None:
            try:
                auc_list.append(roc_auc_score(yt, ys))
            except Exception:
                auc_list.append(np.nan)

    def ci(a):
        return (np.nanpercentile(a, 2.5), np.nanpercentile(a, 97.5))

    out = {
        "sens_CI": ci(sens_list),
        "spec_CI": ci(spec_list),
    }
    if y_score is not None:
        out["auc_CI"] = ci(auc_list)
    return out


In [ ]:
def binary_perf_CI(y_true, y_pred, y_score=None, n_boot=2000):
    """Point estimates + bootstrapped CIs for sens/spec/AUC."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sens = tp/(tp+fn) if (tp+fn) > 0 else np.nan
    spec = tn/(tn+fp) if (tn+fp) > 0 else np.nan
    auc  = roc_auc_score(y_true, y_score) if y_score is not None else np.nan

    boot = bootstrap_binary_metrics(y_true, y_pred, y_score, n_boot=n_boot)

    out = {
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),
        "sensitivity": sens,
        "specificity": spec,
        "AUC": auc,
        "sens_95CI": boot["sens_CI"],
        "spec_95CI": boot["spec_CI"],
    }
    if y_score is not None:
        out["AUC_95CI"] = boot["auc_CI"]
    return out

In [ ]:
def bootstrap_proportion(x, n_boot=2000, seed=1):
    """x is 0/1 array; returns mean and 95% CI via bootstrap."""
    x = np.asarray(x).astype(float)
    rng = np.random.default_rng(seed)
    n = len(x)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        vals.append(x[idx].mean())
    vals = np.array(vals)
    return x.mean(), (np.nanpercentile(vals, 2.5), np.nanpercentile(vals, 97.5))

In [ ]:
# ---------------------------------------------------------------------
# 2. OVERALL PERFORMANCE (entire enriched dataset)
# ---------------------------------------------------------------------
df = result_df.copy()

y_true = df["cancer"].values

overall_rad = binary_perf_CI(
    y_true,
    df["rad_pos"].values,
    y_score=None
)

overall_ai = binary_perf_CI(
    y_true,
    df["ai_pos"].values,
    y_score=df["ai_score"].values
)

print("=== OVERALL PERFORMANCE ===")
print("Radiologist:", overall_rad)
print("AI:", overall_ai)

overall_ctab = pd.crosstab(df["ai_outcome"], df["rad_outcome"])
print("\n4x4 error matrix (AI vs Radiologist):")
print(overall_ctab)

In [ ]:
# ---------------------------------------------------------------------
# 3. STRATUM 1 – CORRECT CASES (radiologist TP + TN)
# ---------------------------------------------------------------------

correct_df = df[df["rad_outcome"].isin(["TP", "TN"])].copy()
print("\n=== STRATUM 1: CORRECT CASES (Rad TP + TN) ===")
print("n =", len(correct_df))

# 3.1 AI performance vs truth on these cases
y_true_correct = correct_df["cancer"].values
ai_correct_perf = binary_perf_CI(
    y_true_correct,
    correct_df["ai_pos"].values,
    y_score=correct_df["ai_score"].values
)
print("AI perf on rad-correct cases:", ai_correct_perf)

# 3.2 Agreement matrix in correct cases (how often AI matches TP/TN)
correct_ctab = pd.crosstab(correct_df["ai_outcome"], correct_df["rad_outcome"])
print("\nAI vs Radiologist outcomes in correct stratum:")
print(correct_ctab)

# 3.3 Optional: density subgroup analysis within correct cases
def subgroup_perf(df_sub, group_col):
    rows = []
    for g, sub in df_sub.groupby(group_col):
        y = sub["cancer"].values
        ai_res = binary_perf_CI(
            y,
            sub["ai_pos"].values,
            y_score=sub["ai_score"].values
        )
        rows.append({
            group_col: g,
            "n": len(sub),
            "ai_sens": ai_res["sensitivity"],
            "ai_sens_CI": ai_res["sens_95CI"],
            "ai_spec": ai_res["specificity"],
            "ai_spec_CI": ai_res["spec_95CI"],
            "ai_auc": ai_res["AUC"],
            "ai_auc_CI": ai_res.get("AUC_95CI", (np.nan, np.nan)),
        })
    return pd.DataFrame(rows)

# If your density column is named differently, change here:
density_correct = subgroup_perf(correct_df, "DensityCategory_std")
print("\nAI performance by density (correct stratum):")
print(density_correct)


In [ ]:
# ---------------------------------------------------------------------
# 4. STRATUM 2 – ERROR CASES (radiologist FP + FN)
# ---------------------------------------------------------------------

error_df = df[df["rad_outcome"].isin(["FP", "FN"])].copy()
print("\n=== STRATUM 2: ERROR CASES (Rad FP + FN) ===")
print("n =", len(error_df))

error_ctab = pd.crosstab(error_df["ai_outcome"], error_df["rad_outcome"])
print("\nAI vs Radiologist outcomes in error stratum:")
print(error_ctab)

# 4.1 AI recall among radiologist FN cancers
rad_FN = error_df[error_df["rad_outcome"] == "FN"].copy()
if not rad_FN.empty:
    ai_recall_FN, ai_recall_FN_CI = bootstrap_proportion(
        (rad_FN["ai_pos"] == 1).astype(int)
    )
    print(f"\nAI recall among radiologist FN cancers: "
          f"{ai_recall_FN:.3f} (95% CI {ai_recall_FN_CI[0]:.3f}–{ai_recall_FN_CI[1]:.3f}) "
          f" [n={len(rad_FN)}]")
else:
    print("\nNo radiologist FN cases in dataset.")

# 4.2 AI specificity among radiologist FP benign recalls
rad_FP = error_df[error_df["rad_outcome"] == "FP"].copy()
if not rad_FP.empty:
    ai_spec_FP, ai_spec_FP_CI = bootstrap_proportion(
        (rad_FP["ai_pos"] == 0).astype(int)
    )
    print(f"AI specificity among radiologist FP benign recalls: "
          f"{ai_spec_FP:.3f} (95% CI {ai_spec_FP_CI[0]:.3f}–{ai_spec_FP_CI[1]:.3f}) "
          f" [n={len(rad_FP)}]")
else:
    print("No radiologist FP cases in dataset.")

# 4.3 Optional: density subgroup analysis within error cases
density_error = subgroup_perf(error_df, "DensityCategory_std")
print("\nAI performance by density (error stratum):")
print(density_error)

In [ ]:
# ---------------------------------------------------------------------
# 5. LOGISTIC MODELS (OPTIONAL, STILL ON FULL DATASET)
#    – FN vs TP among cancers, FP vs TN among benign
# ---------------------------------------------------------------------

# 5.1 FN vs TP (among cancers)

# df_cancer = df[df["cancer"] == 1].copy()
# df_cancer["is_FN"] = (df_cancer["rad_outcome"] == "FN").astype(int)

# # Identify lesion types with both FN=0 and FN=1
# valid_lesions = (
#     df_cancer.groupby("LesionType_coarse")["is_FN"].nunique() == 2
# )

# valid_lesions = valid_lesions[valid_lesions].index.tolist()
# print("Keeping lesion types:", valid_lesions)

# df_cancer2 = df_cancer[df_cancer["LesionType_coarse"].isin(valid_lesions)].copy()

# fn_model = smf.logit(
#     "is_FN ~ PatientAge + C(Density_coarse) + C(LesionType_coarse) + ai_score",
#     data=df_cancer2
# ).fit(maxiter=100, disp=True)

# print(fn_model.summary())

# # ---------------------------------------------------------------------
# Checks 

# import statsmodels.formula.api as smf
# import numpy as np
# import pandas as pd

# df_cancer = df[df["cancer"] == 1].copy()
# df_cancer["is_FN"] = (df_cancer["rad_outcome"] == "FN").astype(int)

# mod = smf.logit(
#     "is_FN ~ PatientAge + C(Density_coarse) + C(LesionType_coarse) + ai_score",
#     data=df_cancer
# )

# X = mod.exog
# print("X shape:", X.shape)
# print("Matrix rank:", np.linalg.matrix_rank(X))

# print("\nDensity vs FN:")
# print(pd.crosstab(df_cancer["Density_coarse"], df_cancer["is_FN"]))

# print("\nLesionType vs FN:")
# print(pd.crosstab(df_cancer["LesionType_coarse"], df_cancer["is_FN"]))

In [ ]:
# # ---------------------------------------------------------------------
# # 5. LOGISTIC MODELS (OPTIONAL, STILL ON FULL DATASET)
# #    – FN vs TP among cancers, FP vs TN among benign
# # ---------------------------------------------------------------------

# # 5.1 FN vs TP (among cancers)
# df_cancer = df[df["cancer"] == 1].copy()
# df_cancer["is_FN"] = (df_cancer["rad_outcome"] == "FN").astype(int)

# fn_model = smf.logit(
#     "is_FN ~ PatientAge + C(DensityCategory_std) + C(LesionType_cat) + ai_score",
#     data=df_cancer
# ).fit(maxiter=50, disp=False)

# print("\nLogit FN vs TP among cancers (interpret with care if quasi-separation):")
# print(fn_model.summary())

# # 5.2 FP vs TN (among benign)
# df_benign = df[df["cancer"] == 0].copy()
# df_benign["is_FP"] = (df_benign["rad_outcome"] == "FP").astype(int)

# fp_model = smf.logit(
#     "is_FP ~ PatientAge + C(DensityCategory_std) + C(LesionType_cat) + ai_score",
#     data=df_benign
# ).fit(maxiter=50, disp=False)

# print("\nLogit FP vs TN among benign (interpret with care if quasi-separation):")
# print(fp_model.summary())

In [ ]:
# # ---------------------------------------------------------------------
# # 5. LOGISTIC MODELS (OPTIONAL, STILL ON FULL DATASET)
# #    – FN vs TP among cancers, FP vs TN among benign
# # ---------------------------------------------------------------------

# # 5.1 FN vs TP (among cancers)
# df_cancer = df[df["cancer"] == 1].copy()
# df_cancer["is_FN"] = (df_cancer["rad_outcome"] == "FN").astype(int)

# fn_model = smf.logit(
#     "is_FN ~ PatientAge + C(Density_coarse) + C(LesionType_coarse) + ai_score",
#     data=df_cancer
# ).fit(maxiter=50, disp=False)

# print("\nLogit FN vs TP among cancers (interpret with care if quasi-separation):")
# print(fn_model.summary())

# # 5.2 FP vs TN (among benign)
# df_benign = df[df["cancer"] == 0].copy()
# df_benign["is_FP"] = (df_benign["rad_outcome"] == "FP").astype(int)

# fp_model = smf.logit(
#     "is_FP ~ PatientAge + C(Density_coarse) + C(LesionType_coarse) + ai_score",
#     data=df_benign
# ).fit(maxiter=50, disp=False)

# print("\nLogit FP vs TN among benign (interpret with care if quasi-separation):")
# print(fp_model.summary())

In [ ]:
# df_cancer.columns.to_list()

In [ ]:
#----------------------------------------------------
# Helper functions
#----------------------------------------------------

def confusion_counts(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return {"TP": tp, "TN": tn, "FP": fp, "FN": fn}

def bootstrap_ci_proportion(successes: int, total: int, n_boot: int = 10000,
                            alpha: float = 0.05, random_state: int = 42) -> Tuple[float, float]:
    rng = np.random.default_rng(random_state)
    p_hat = successes / total if total > 0 else np.nan
    if total == 0:
        return (np.nan, np.nan)
    boot_samples = rng.binomial(total, p_hat, size=n_boot) / total
    lower = np.percentile(boot_samples, 100 * alpha / 2)
    upper = np.percentile(boot_samples, 100 * (1 - alpha / 2))
    return lower, upper

def bootstrap_ci_auc(y_true: np.ndarray, scores: np.ndarray,
                     n_boot: int = 10000, alpha: float = 0.05,
                     random_state: int = 42) -> Tuple[float, float, float]:
    rng = np.random.default_rng(random_state)
    auc_list = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        try:
            auc = roc_auc_score(y_true[idx], scores[idx])
            auc_list.append(auc)
        except ValueError:
            # Happens if all labels in resample are the same
            continue
    auc_arr = np.array(auc_list)
    auc_mean = np.mean(auc_arr)
    lower = np.percentile(auc_arr, 100 * alpha / 2)
    upper = np.percentile(auc_arr, 100 * (1 - alpha / 2))
    return auc_mean, lower, upper

def bayes_adjust_ppv_npv(sens: float, spec: float, prevalence: float = 0.005) -> Tuple[float, float]:
    """Return PPV and NPV at a given disease prevalence using Bayes' theorem."""
    ppv = (sens * prevalence) / (sens * prevalence + (1 - spec) * (1 - prevalence))
    npv = (spec * (1 - prevalence)) / ((1 - sens) * prevalence + spec * (1 - prevalence))
    return ppv, npv

In [ ]:
df = result_df.copy()
df['stratum'] = ['correct' if i in ['TN', 'TP'] else 'error' for i in df['Group']]

In [ ]:
#----------------------------------------------------
# Table 2: global performance (radiologist and AI)
#----------------------------------------------------

# Radiologist predicted labels
df["rad_pred"] = df["rad_pos"].astype(int)
df["ai_pred"] = df["ai_pos"].astype(int)

true = df["cancer"].values
rad_pred = df["rad_pred"].values
ai_pred = df["ai_pred"].values
ai_scores = df["ai_score"].values

rad_counts = confusion_counts(true, rad_pred)
ai_counts = confusion_counts(true, ai_pred)

def sens_spec(counts: dict) -> Tuple[float, float]:
    tp, tn, fp, fn = counts["TP"], counts["TN"], counts["FP"], counts["FN"]
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return sens, spec

rad_sens, rad_spec = sens_spec(rad_counts)
ai_sens, ai_spec = sens_spec(ai_counts)

# Bootstrap CIs for sensitivity and specificity
rad_sens_ci = bootstrap_ci_proportion(rad_counts["TP"], rad_counts["TP"] + rad_counts["FN"])
rad_spec_ci = bootstrap_ci_proportion(rad_counts["TN"], rad_counts["TN"] + rad_counts["FP"])

ai_sens_ci = bootstrap_ci_proportion(ai_counts["TP"], ai_counts["TP"] + ai_counts["FN"])
ai_spec_ci = bootstrap_ci_proportion(ai_counts["TN"], ai_counts["TN"] + ai_counts["FP"])

# AUC and bootstrap CI for AI
ai_auc, ai_auc_low, ai_auc_high = bootstrap_ci_auc(true, ai_scores)

# Adjusted PPV/NPV at 0.5% prevalence
rad_ppv_adj, rad_npv_adj = bayes_adjust_ppv_npv(rad_sens, rad_spec, prevalence=0.005)
ai_ppv_adj, ai_npv_adj = bayes_adjust_ppv_npv(ai_sens, ai_spec, prevalence=0.005)

table2 = pd.DataFrame({
    "Metric": [
        "TP", "TN", "FP", "FN",
        "Sensitivity", "Specificity", "AUC",
        "Adj_PPV_0.5%", "Adj_NPV_0.5%"
    ],
    "Radiologist": [
        rad_counts["TP"],
        rad_counts["TN"],
        rad_counts["FP"],
        rad_counts["FN"],
        f"{rad_sens*100:.1f} ({rad_sens_ci[0]*100:.1f}–{rad_sens_ci[1]*100:.1f})",
        f"{rad_spec*100:.1f} ({rad_spec_ci[0]*100:.1f}–{rad_spec_ci[1]*100:.1f})",
        "–",
        f"{rad_ppv_adj*100:.2f}",
        f"{rad_npv_adj*100:.2f}",
    ],
    "AI": [
        ai_counts["TP"],
        ai_counts["TN"],
        ai_counts["FP"],
        ai_counts["FN"],
        f"{ai_sens*100:.1f} ({ai_sens_ci[0]*100:.1f}–{ai_sens_ci[1]*100:.1f})",
        f"{ai_spec*100:.1f} ({ai_spec_ci[0]*100:.1f}–{ai_spec_ci[1]*100:.1f})",
        f"{ai_auc:.3f} ({ai_auc_low:.3f}–{ai_auc_high:.3f})",
        f"{ai_ppv_adj*100:.2f}",
        f"{ai_npv_adj*100:.2f}",
    ]
})

print("Table 2: global performance")
print(table2.to_string(index=False))

In [ ]:
# #----------------------------------------------------
# # Overall AI performance
# #----------------------------------------------------

# y_true = df["cancer"].values
# y_pred_ai = df["ai_pos"].values
# scores_ai = df["ai_score"].values

# overall_counts = confusion_counts(y_true, y_pred_ai)
# overall_sens, overall_spec = sens_spec(overall_counts)
# overall_sens_ci = bootstrap_ci_proportion(overall_counts["TP"], overall_counts["TP"] + overall_counts["FN"])
# overall_spec_ci = bootstrap_ci_proportion(overall_counts["TN"], overall_counts["TN"] + overall_counts["FP"])
# overall_auc, overall_auc_low, overall_auc_high = bootstrap_ci_auc(y_true, scores_ai)

# #----------------------------------------------------
# # Stratum 1: Radiologist-correct
# #----------------------------------------------------

# df_correct = df[df["stratum"] == "correct"].copy()
# y_true_corr = df_correct["cancer"].values
# y_pred_corr = df_correct["ai_pos"].values
# scores_corr = df_correct["ai_score"].values

# corr_counts = confusion_counts(y_true_corr, y_pred_corr)
# corr_sens, corr_spec = sens_spec(corr_counts)
# corr_sens_ci = bootstrap_ci_proportion(corr_counts["TP"], corr_counts["TP"] + corr_counts["FN"])
# corr_spec_ci = bootstrap_ci_proportion(corr_counts["TN"], corr_counts["TN"] + corr_counts["FP"])
# corr_auc, corr_auc_low, corr_auc_high = bootstrap_ci_auc(y_true_corr, scores_corr)

# #----------------------------------------------------
# # Stratum 2: Radiologist-error
# #   Radiologist FN cancers: cancer == 1 & rad_outcome == "FN"
# #   Radiologist FP benign:  cancer == 0 & rad_outcome == "FP"
# #----------------------------------------------------

# df_error = df[df["stratum"] == "error"].copy()

# rad_fn = df_error[(df_error["cancer"] == 1) & (df_error["rad_outcome"] == "FN")]
# rad_fp = df_error[(df_error["cancer"] == 0) & (df_error["rad_outcome"] == "FP")]

# # AI recall among radiologist FN cancers
# rescued = np.sum(rad_fn["ai_pos"] == 1)
# n_rad_fn = len(rad_fn)
# rescued_rate = rescued / n_rad_fn if n_rad_fn > 0 else np.nan
# rescued_ci = bootstrap_ci_proportion(rescued, n_rad_fn) if n_rad_fn > 0 else (np.nan, np.nan)

# # AI specificity among radiologist FP benign recalls (down-classification)
# avoided = np.sum(rad_fp["ai_pos"] == 0)
# n_rad_fp = len(rad_fp)
# avoided_rate = avoided / n_rad_fp if n_rad_fp > 0 else np.nan
# avoided_ci = bootstrap_ci_proportion(avoided, n_rad_fp) if n_rad_fp > 0 else (np.nan, np.nan)

# #----------------------------------------------------
# # Assemble Table 2 (overall + strata)
# #----------------------------------------------------

# rows = []

# # Overall
# rows.append({
#     "Stratum": "Overall",
#     "Metric": "AI sensitivity",
#     "n": overall_counts["TP"] + overall_counts["FN"],
#     "Estimate": overall_sens,
#     "CI_lower": overall_sens_ci[0],
#     "CI_upper": overall_sens_ci[1]
# })
# rows.append({
#     "Stratum": "Overall",
#     "Metric": "AI specificity",
#     "n": overall_counts["TN"] + overall_counts["FP"],
#     "Estimate": overall_spec,
#     "CI_lower": overall_spec_ci[0],
#     "CI_upper": overall_spec_ci[1]
# })
# rows.append({
#     "Stratum": "Overall",
#     "Metric": "AI AUC",
#     "n": len(df),
#     "Estimate": overall_auc,
#     "CI_lower": overall_auc_low,
#     "CI_upper": overall_auc_high
# })

# # Radiologist-concordant (correct) stratum
# rows.append({
#     "Stratum": "Radiologist-concordant",
#     "Metric": "AI sensitivity among cancers",
#     "n": corr_counts["TP"] + corr_counts["FN"],
#     "Estimate": corr_sens,
#     "CI_lower": corr_sens_ci[0],
#     "CI_upper": corr_sens_ci[1]
# })
# rows.append({
#     "Stratum": "Radiologist-concordant",
#     "Metric": "AI specificity among benign",
#     "n": corr_counts["TN"] + corr_counts["FP"],
#     "Estimate": corr_spec,
#     "CI_lower": corr_spec_ci[0],
#     "CI_upper": corr_spec_ci[1]
# })
# rows.append({
#     "Stratum": "Radiologist-concordant",
#     "Metric": "AI AUC",
#     "n": len(df_correct),
#     "Estimate": corr_auc,
#     "CI_lower": corr_auc_low,
#     "CI_upper": corr_auc_high
# })

# # Radiologist-error stratum
# rows.append({
#     "Stratum": "Radiologist-error",
#     "Metric": "AI recall among radiologist FN cancers",
#     "n": n_rad_fn,
#     "Estimate": rescued_rate,
#     "CI_lower": rescued_ci[0],
#     "CI_upper": rescued_ci[1]
# })
# rows.append({
#     "Stratum": "Radiologist-error",
#     "Metric": "AI specificity among radiologist FP benign recalls",
#     "n": n_rad_fp,
#     "Estimate": avoided_rate,
#     "CI_lower": avoided_ci[0],
#     "CI_upper": avoided_ci[1]
# })

# table2 = pd.DataFrame(rows)

# # Format as percentages for display
# table2_display = table2.copy()
# is_auc = table2_display["Metric"].str.contains("AUC")
# table2_display.loc[~is_auc, "Estimate"] = (table2_display.loc[~is_auc, "Estimate"] * 100).round(1)
# table2_display.loc[~is_auc, "CI_lower"] = (table2_display.loc[~is_auc, "CI_lower"] * 100).round(1)
# table2_display.loc[~is_auc, "CI_upper"] = (table2_display.loc[~is_auc, "CI_upper"] * 100).round(1)

# print("\nTable 2: Overall and stratum-specific AI performance")
# print(table2_display.to_string(index=False))

In [ ]:
df_correct = df[df["stratum"] == "correct"].copy()
df_error = df[df["stratum"] == "error"].copy()

# In the correct stratum, cancers are radiologist TP; benign are radiologist TN
correct_true = df_correct["cancer"].values
correct_ai_pred = df_correct["ai_pred"].values
correct_ai_scores = df_correct["ai_score"].values

correct_counts = confusion_counts(correct_true, correct_ai_pred)
correct_sens, correct_spec = sens_spec(correct_counts)
correct_sens_ci = bootstrap_ci_proportion(correct_counts["TP"], correct_counts["TP"] + correct_counts["FN"])
correct_spec_ci = bootstrap_ci_proportion(correct_counts["TN"], correct_counts["TN"] + correct_counts["FP"])
correct_auc, correct_auc_low, correct_auc_high = bootstrap_ci_auc(correct_true, correct_ai_scores)

table3 = pd.DataFrame({
    "Metric": ["Sensitivity", "Specificity", "AUC", "FN_count", "FP_count"],
    "Value": [
        f"{correct_sens*100:.1f} ({correct_sens_ci[0]*100:.1f}–{correct_sens_ci[1]*100:.1f})",
        f"{correct_spec*100:.1f} ({correct_spec_ci[0]*100:.1f}–{correct_spec_ci[1]*100:.1f})",
        f"{correct_auc:.3f} ({correct_auc_low:.3f}–{correct_auc_high:.3f})",
        correct_counts["FN"],
        correct_counts["FP"]
    ]
})

print("\nTable 3: AI performance in radiologist-correct stratum")
print(table3.to_string(index=False))

In [ ]:
# In the error stratum:
# Radiologist FN: cancer == 1 & rad_pos == 0
# Radiologist FP: cancer == 0 & rad_pos == 1

error_rad_fn = df_error[(df_error["cancer"] == 1) & (df_error["rad_pos"] == 0)]
error_rad_fp = df_error[(df_error["cancer"] == 0) & (df_error["rad_pos"] == 1)]

rescued_cancers = np.sum(error_rad_fn["ai_pred"] == 1)
total_fn = len(error_rad_fn)
avoided_recalls = np.sum(error_rad_fp["ai_pred"] == 0)
total_fp = len(error_rad_fp)

rescued_rate = rescued_cancers / total_fn if total_fn > 0 else np.nan
rescued_ci = bootstrap_ci_proportion(rescued_cancers, total_fn)

avoided_rate = avoided_recalls / total_fp if total_fp > 0 else np.nan
avoided_ci = bootstrap_ci_proportion(avoided_recalls, total_fp)

table4 = pd.DataFrame({
    "Scenario": [
        "Radiologist FN cancers recalled by AI",
        "Radiologist FN cancers missed by both",
        "Radiologist FP benign cases down-classified by AI",
        "Radiologist FP benign cases recalled by both"
    ],
    "Estimate": [
        f"{rescued_rate*100:.1f} ({rescued_ci[0]*100:.1f}–{rescued_ci[1]*100:.1f})",
        f"{(1-rescued_rate)*100:.1f}",
        f"{avoided_rate*100:.1f} ({avoided_ci[0]*100:.1f}–{avoided_ci[1]*100:.1f})",
        f"{(1-avoided_rate)*100:.1f}",
    ],
    "Count": [
        f"{rescued_cancers}/{total_fn}",
        f"{total_fn-rescued_cancers}/{total_fn}",
        f"{avoided_recalls}/{total_fp}",
        f"{total_fp-avoided_recalls}/{total_fp}",
    ]
})

print("\nTable 4: error-stratum performance")
print(table4.to_string(index=False))

In [ ]:
# Derive categorical outcomes for radiologist and AI
def outcome_label(y_true, y_pred):
    if y_true == 1 and y_pred == 1:
        return "TP"
    elif y_true == 0 and y_pred == 0:
        return "TN"
    elif y_true == 0 and y_pred == 1:
        return "FP"
    elif y_true == 1 and y_pred == 0:
        return "FN"
    return "NA"

df["rad_outcome"] = [outcome_label(t, p) for t, p in zip(df["cancer"], df["rad_pred"])]
df["ai_outcome"] = [outcome_label(t, p) for t, p in zip(df["cancer"], df["ai_pred"])]

matrix = pd.crosstab(df["ai_outcome"], df["rad_outcome"])
print("\nTable 5: 4x4 AI vs Radiologist outcomes")
print(matrix)

In [ ]:
# ---------------------------------------------------------------------
# 5. LOGISTIC MODELS (OPTIONAL, STILL ON FULL DATASET)
#    – FN vs TP among cancers, FP vs TN among benign
# ---------------------------------------------------------------------

# 5.1 FN vs TP (among cancers)
# df_cancer = df[df["cancer"] == 1].copy()
# df_cancer["is_FN"] = (df_cancer["rad_outcome"] == "FN").astype(int)

# fn_model = smf.logit(
#     "is_FN ~ PatientAge + C(Density_coarse) + C(LesionType_coarse) + ai_score",
#     data=df_cancer
# ).fit(maxiter=50, disp=False)

# print("\nLogit FN vs TP among cancers (interpret with care if quasi-separation):")
# print(fn_model.summary())

# 5.2 FP vs TN (among benign)
# df_benign = df[df["cancer"] == 0].copy()
# df_benign["is_FP"] = (df_benign["rad_outcome"] == "FP").astype(int)

# fp_model = smf.logit(
#     "is_FP ~ PatientAge + C(Density_coarse) + C(LesionType_coarse) + ai_score",
#     data=df_benign
# ).fit(maxiter=50, disp=False)

# print("\nLogit FP vs TN among benign (interpret with care if quasi-separation):")
# print(fp_model.summary())

In [ ]:
# ==========================
# 1. CANCER GROUP: false negatives (is_FN)
# ==========================

df_cancer = df[df["cancer"] == 1].copy()
df_cancer["is_FN"] = (df_cancer["rad_outcome"] == "FN").astype(int)

# Keep only lesion types with both FN=0 and FN=1
valid_lesions_fn = (
    df_cancer.groupby("LesionType_coarse")["is_FN"].nunique() == 2
)
valid_lesions_fn = valid_lesions_fn[valid_lesions_fn].index.tolist()
print("Keeping lesion types for FN model:", valid_lesions_fn)

df_cancer2 = df_cancer[df_cancer["LesionType_coarse"].isin(valid_lesions_fn)].copy()

fn_model = smf.logit(
    "is_FN ~ PatientAge + C(Density_coarse) + C(LesionType_coarse) + ai_score",
    data=df_cancer2
).fit(maxiter=100, disp=True)

print("\n=== Logistic regression for radiologist FN among cancers ===")
print(fn_model.summary())

In [ ]:
# ==========================
# 2. BENIGN GROUP: false positives (is_FP)
# ==========================

df_benign = df[df["cancer"] == 0].copy()
df_benign["is_FP"] = (df_benign["rad_outcome"] == "FP").astype(int)

# Keep only lesion types with both FP=0 and FP=1
valid_lesions_fp = (
    df_benign.groupby("LesionType_coarse")["is_FP"].nunique() == 2
)
valid_lesions_fp = valid_lesions_fp[valid_lesions_fp].index.tolist()
print("\nKeeping lesion types for FP model:", valid_lesions_fp)

df_benign2 = df_benign[df_benign["LesionType_coarse"].isin(valid_lesions_fp)].copy()

fp_model = smf.logit(
    "is_FP ~ PatientAge + C(Density_coarse) + C(LesionType_coarse) + ai_score",
    data=df_benign2
).fit(maxiter=100, disp=True)

print("\n=== Logistic regression for radiologist FP among benign ===")
print(fp_model.summary())

In [ ]:
# ROC curve for AI, overall
fpr, tpr, _ = roc_curve(true, ai_scores)
plt.figure()
plt.plot(fpr, tpr, label=f"AI (AUC = {ai_auc:.3f})")
plt.plot([0,1], [0,1], linestyle="--", color="grey")
plt.xlabel("1 - Specificity")
plt.ylabel("Sensitivity")
plt.title("ROC curve for FxMammo (overall)")
plt.legend()
plt.tight_layout()
plt.show()

# Heatmap of 4x4 matrix
plt.figure()
im = plt.imshow(matrix.values, cmap="Blues")
plt.colorbar(im)
plt.xticks(range(matrix.shape[1]), matrix.columns)
plt.yticks(range(matrix.shape[0]), matrix.index)
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        plt.text(j, i, matrix.values[i, j],
                 ha="center", va="center", color="black")
plt.xlabel("Radiologist outcome")
plt.ylabel("AI outcome")
plt.title("Radiologist–AI discordance matrix")
plt.tight_layout()
plt.show()

# Bar plot: AI recall among radiologist FN, by density
fn_by_density = (
    error_rad_fn
    .groupby("Density_std")["ai_pred"]
    .agg(["sum", "count"])
    .reset_index()
)
fn_by_density["rate"] = fn_by_density["sum"] / fn_by_density["count"]

plt.figure()
plt.bar(fn_by_density["Density_std"], fn_by_density["rate"])
plt.ylim(0, 1)
plt.ylabel("AI recall rate among radiologist FN cancers")
plt.xlabel("Breast density")
plt.title("AI performance on radiologist false-negative cancers by density")
plt.tight_layout()
plt.show()

In [ ]:




#----------------------------------------------------
# Assume df is your main dataframe
#----------------------------------------------------

# Columns assumed:
# df["cancer"] in {0,1}
# df["rad_pos"] in {0,1}
# df["ai_pos"] in {0,1}
# df["ai_score"] float
# df["stratum"] in {"correct", "error"}
# df["Density"] in {"A","B","C","D"}
# df["LesionType_coarse"], df["PatientAge"]

# Example dummy line (replace with your actual loading)
# df = pd.read_csv("your_dataset.csv")



#----------------------------------------------------
# Stratum-specific performance (Tables 3 and 4)
#----------------------------------------------------





#----------------------------------------------------
# Table 5: 4x4 radiologist–AI matrix
#----------------------------------------------------



#----------------------------------------------------
# Table 6: logistic regression for false-negative cancers
#----------------------------------------------------

# Restrict to cancers only for FN vs TP model
df_cancer = df[df["cancer"] == 1].copy()
df_cancer["is_FN"] = (df_cancer["ai_outcome"] == "FN").astype(int)

# Example: coarsen density and lesion type if needed
# df_cancer["Density_coarse"] = df_cancer["Density"].map({...})

formula = "is_FN ~ C(Density) + C(LesionType_coarse) + PatientAge + ai_score"
logit_model = smf.logit(formula, data=df_cancer).fit()
print("\nLogistic regression for AI false-negatives vs true-positives")
print(logit_model.summary())